In [ ]:
import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader

from src.data_loader import load_pkl, load_or_build_cache
from src.features import extract_sequence_features
from src.advanced_models import (LateFusionModel, EarlyFusionModel,
                                   CrossModalAttentionFusion, count_parameters)
from src.train import (MOSEIBimodalDataset, fit_bimodal, evaluate_bimodal,
                        compute_sentiment_class_weights, compute_emotion_pos_weights)

pkl_data = load_pkl()
label_cache = load_or_build_cache()
seq_features = extract_sequence_features(pkl_data)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

dummy_text = torch.randn(4, 50, 300).to(device)
dummy_audio = torch.randn(4, 50, 74).to(device)
dummy_text_mask = torch.ones(4, 50, dtype=torch.bool).to(device)
dummy_audio_mask = torch.ones(4, 50, dtype=torch.bool).to(device)
dummy_text_mask[2, 30:] = False
dummy_audio_mask[3, 20:] = False

for ModelCls, name in [
    (LateFusionModel, "LateFusion"),
    (EarlyFusionModel, "EarlyFusion"),
    (CrossModalAttentionFusion, "CrossModalAttention"),
]:
    model = ModelCls(text_dim=300, audio_dim=74).to(device)
    sent_out, emo_out = model((dummy_text, dummy_audio), (dummy_text_mask, dummy_audio_mask))
    print(f"  {name:22s} | sent={tuple(sent_out.shape)} emo={tuple(emo_out.shape)} | params={count_parameters(model):,}")

In [ ]:
import os
import pandas as pd

os.makedirs("../experiments", exist_ok=True)

text_train, text_mask_train = seq_features["text"]["train"]["X"], seq_features["text"]["train"]["mask"]
audio_train, audio_mask_train = seq_features["audio"]["train"]["X"], seq_features["audio"]["train"]["mask"]
text_valid, text_mask_valid = seq_features["text"]["valid"]["X"], seq_features["text"]["valid"]["mask"]
audio_valid, audio_mask_valid = seq_features["audio"]["valid"]["X"], seq_features["audio"]["valid"]["mask"]
text_test, text_mask_test = seq_features["text"]["test"]["X"], seq_features["text"]["test"]["mask"]
audio_test, audio_mask_test = seq_features["audio"]["test"]["X"], seq_features["audio"]["test"]["mask"]

train_ds = MOSEIBimodalDataset(text_train, text_mask_train, audio_train, audio_mask_train,
                                label_cache["train"]["sentiment_class"],
                                label_cache["train"]["emotion_binary"],
                                label_cache["train"]["mask"])
valid_ds = MOSEIBimodalDataset(text_valid, text_mask_valid, audio_valid, audio_mask_valid,
                                label_cache["valid"]["sentiment_class"],
                                label_cache["valid"]["emotion_binary"],
                                label_cache["valid"]["mask"])
test_ds = MOSEIBimodalDataset(text_test, text_mask_test, audio_test, audio_mask_test,
                               label_cache["test"]["sentiment_class"],
                               label_cache["test"]["emotion_binary"],
                               label_cache["test"]["mask"])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

sent_weights = compute_sentiment_class_weights(label_cache["train"]["sentiment_class"])
emo_weights = compute_emotion_pos_weights(label_cache["train"]["emotion_binary"], label_cache["train"]["mask"])

FUSION_CONFIGS = [
    ("Late fusion", LateFusionModel),
    ("Early fusion", EarlyFusionModel),
    ("Cross-modal attention", CrossModalAttentionFusion),
]

fusion_results = []
fusion_histories = {}
fusion_models = {}

for name, ModelCls in FUSION_CONFIGS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")

    model = ModelCls(text_dim=300, audio_dim=74, d_model=128, num_layers=2, dropout=0.1)
    n_params = count_parameters(model)

    model, history = fit_bimodal(model, train_loader, valid_loader, device,
                                  sent_weights, emo_weights, epochs=30, lr=5e-4,
                                  patience=5, verbose=False)

    sm_test, em_test = evaluate_bimodal(model, test_loader, device)

    fusion_results.append({
        "model": name,
        "modalities": "text + audio",
        "n_params": n_params,
        "epochs_trained": len(history),
        "sentiment_macro_f1": round(sm_test["macro_f1_3class"], 3),
        "sentiment_acc2": round(sm_test["acc2"], 3),
        "emotion_macro_f1": round(em_test["macro_f1"], 3),
        "emotion_mean_auc": round(em_test["mean_auc"], 3),
    })
    fusion_histories[name] = history
    fusion_models[name] = model

    print(f"TEST: sentiment macro-F1={sm_test['macro_f1_3class']:.3f} | "
          f"emotion macro-F1={em_test['macro_f1']:.3f} | epochs={len(history)}")

fusion_df = pd.DataFrame(fusion_results)
fusion_df.to_csv("../experiments/results_fusion.csv", index=False)
fusion_df

In [ ]:
comparison_rows = [
    {"model": "Best baseline (GloVe FFN)", "modalities": "text only",
     "n_params": 111113, "epochs_trained": 17,
     "sentiment_macro_f1": 0.589, "sentiment_acc2": 0.794,
     "emotion_macro_f1": 0.409, "emotion_mean_auc": 0.685},
    {"model": "Transformer tuned", "modalities": "text only",
     "n_params": 321161, "epochs_trained": 8,
     "sentiment_macro_f1": 0.602, "sentiment_acc2": None,
     "emotion_macro_f1": 0.413, "emotion_mean_auc": 0.686},
    {"model": "Transformer (default)", "modalities": "audio only",
     "n_params": 292233, "epochs_trained": 11,
     "sentiment_macro_f1": 0.444, "sentiment_acc2": 0.686,
     "emotion_macro_f1": 0.362, "emotion_mean_auc": 0.628},
]

full_comparison = pd.concat([pd.DataFrame(comparison_rows), fusion_df], ignore_index=True)
full_comparison.to_csv("../experiments/final_comparison.csv", index=False)
full_comparison

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import os

os.makedirs("../figures/architectures", exist_ok=True)

GRAY_FILL, GRAY_EDGE = "#F1EFE8", "#5F5E5A"
PURPLE_FILL, PURPLE_EDGE = "#EEEDFE", "#534AB7"
TEAL_FILL, TEAL_EDGE = "#E1F5EE", "#0F6E56"

def box(ax, x, y, w, h, text, color, edge, fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.06",
                                     linewidth=1.1, edgecolor=edge, facecolor=color)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize, fontweight="bold")

def arrow(ax, x1, y1, x2, y2, color="#5F5E5A"):
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=12,
                          linewidth=1.1, color=color)
    ax.add_patch(a)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("(a) Late fusion", fontsize=12, fontweight="bold")
box(ax, 0.5, 8, 4, 1, "Text\nTransformer", PURPLE_FILL, PURPLE_EDGE)
box(ax, 5.5, 8, 4, 1, "Audio\nTransformer", TEAL_FILL, TEAL_EDGE)
arrow(ax, 2.5, 8, 2.5, 6.8)
arrow(ax, 7.5, 8, 7.5, 6.8)
box(ax, 1, 5.5, 8, 1, "Конкатенация (128+128)", GRAY_FILL, GRAY_EDGE)
arrow(ax, 5, 5.5, 5, 4.3)
box(ax, 2.5, 3, 5, 1, "Multitask head", GRAY_FILL, GRAY_EDGE)

ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("(b) Early fusion", fontsize=12, fontweight="bold")
box(ax, 0.5, 8, 4, 1, "Text\nsequence", PURPLE_FILL, PURPLE_EDGE)
box(ax, 5.5, 8, 4, 1, "Audio\nsequence", TEAL_FILL, TEAL_EDGE)
arrow(ax, 2.5, 8, 4.3, 6.8)
arrow(ax, 7.5, 8, 5.7, 6.8)
box(ax, 1, 5.5, 8, 1, "Конкатенация по шагу (374d)", GRAY_FILL, GRAY_EDGE)
arrow(ax, 5, 5.5, 5, 4.3)
box(ax, 1.5, 3, 7, 1, "Общий Transformer encoder", GRAY_FILL, GRAY_EDGE)
arrow(ax, 5, 3, 5, 1.8)
box(ax, 2.5, 0.5, 5, 1, "Multitask head", GRAY_FILL, GRAY_EDGE)

ax = axes[2]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("(c) Cross-modal attention", fontsize=12, fontweight="bold")
box(ax, 0.5, 8, 4, 1, "Text\nself-attn", PURPLE_FILL, PURPLE_EDGE)
box(ax, 5.5, 8, 4, 1, "Audio\nself-attn", TEAL_FILL, TEAL_EDGE)
arrow(ax, 4, 8, 6, 6.8, color=TEAL_EDGE)
arrow(ax, 6, 8, 4, 6.8, color=PURPLE_EDGE)
box(ax, 0.5, 5.5, 4, 1, "Text + audio context", PURPLE_FILL, PURPLE_EDGE, fontsize=8)
box(ax, 5.5, 5.5, 4, 1, "Audio + text context", TEAL_FILL, TEAL_EDGE, fontsize=8)
arrow(ax, 2.5, 5.5, 4.3, 4.3)
arrow(ax, 7.5, 5.5, 5.7, 4.3)
box(ax, 2.5, 3, 5, 1, "Конкатенация + head", GRAY_FILL, GRAY_EDGE)

plt.tight_layout()
plt.savefig("../figures/architectures/fusion_architectures.png", dpi=200, bbox_inches="tight")
plt.show()

Реализованы три стратегии объединения текстовой и акустической модальностей: late fusion (независимые Transformer-энкодеры с конкатенацией представлений), early fusion (конкатенация признаков на уровне временного шага с последующим общим энкодером) и cross-modal attention fusion (раздельные self-attention энкодеры с дополнительным слоем взаимного cross-attention между модальностями). Все три модели использовали гиперпараметры, подобранные для unimodal Transformer в разделе 5 (d_model=128, num_layers=2, dropout=0.1, lr=5·10⁻⁴).
Результаты демонстрируют разнонаправленный эффект объединения модальностей в зависимости от решаемой задачи. По задаче классификации sentiment ни одна из fusion-конфигураций не превзошла лучший unimodal-результат (text-only Transformer, macro-F1 = 0,602): cross-modal attention показал близкий результат (0,600), early fusion — несколько ниже (0,597), late fusion — заметно ниже (0,577), уступив даже исходному baseline. По задаче распознавания эмоций (emotion) картина обратная: все три fusion-модели превзошли лучший unimodal-результат (0,413), с максимумом у late fusion (macro-F1 = 0,424, mean-AUC = 0,705).
Наблюдаемое расхождение интерпретируется следующим образом: акустическая модальность несёт ограниченную дополнительную информацию для классификации общей валентности высказывания, поскольку текстовая модальность уже содержит большую часть необходимого сигнала, тогда как добавление аудио вносит шум, ослабляющий сентимент-сигнал. Для задачи распознавания конкретных эмоций, напротив, просодические характеристики голоса (см. раздел 4, где Prosody+VQ превосходил MFCC) предоставляют различительную информацию, недоступную из текста — в частности, способность отличать эмоции со схожей валентностью, но разной интонационной выраженностью (например, anger и disgust).
Среди трёх стратегий fusion cross-modal attention продемонстрировал наиболее сбалансированный результат по обеим задачам, тогда как late fusion — наименее эффективную интеграцию модальностей на задаче sentiment при лучшем результате на emotion. Это согласуется с ожиданием, что взаимодействие модальностей на более раннем этапе обработки (early и cross-modal fusion) позволяет точнее настроить вклад каждой модальности в зависимости от задачи, чем простое объединение независимо обученных представлений.
Полученные результаты определяют практическую рекомендацию: для задачи классификации сентимента на данном корпусе достаточно одной текстовой модальности с настроенным Transformer-энкодером; для задачи распознавания эмоций целесообразно использовать мультимодальную архитектуру, предпочтительно late или cross-modal fusion.